# Advanced Merges, Duplicate Keys & Data Quality

In [2]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    'customer_id': [101, 102, 103, 104, 105],
    'customer_name': ['Aman', 'Riya', 'Kabir', 'Neha', 'Arjun'],
    'city': ['Delhi', 'Mumbai', 'Delhi', 'Pune', 'Mumbai']
})

orders = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1005, 1006],
    'customer_id': [101, 102, 101, 104, 106, 103],
    'sales': [2500, 1800, 3200, 1500, 4000, 2700],
    'status': [
        'Completed', 'Cancelled', 'Completed',
        'Completed', 'Completed', 'Cancelled'
    ]
})

customer_updates = pd.DataFrame({
    'customer_id': [101, 102, 102, 104, 106],
    'city': ['Delhi', 'Mumbai', 'Mumbai', 'Pune', 'Jaipur'],
    'membership': ['Gold', 'Silver', 'Gold', 'Bronze', 'Silver']
})

product_sales = pd.DataFrame({
    'product_id': [1, 1, 2, 3, 3],
    'category': ['Electronics', 'Electronics', 'Clothing',
                 'Books', 'Books'],
    'sales': [2500, 3200, 1800, 400, 700]
})

product_info = pd.DataFrame({
    'product_id': [1, 2, 3, 4],
    'product_name': ['Laptop', 'Shirt', 'Book', 'Headphones'],
    'category': ['Electronics', 'Clothing', 'Books', 'Electronics']
})

#### Q1. Detect duplicate keys

Check whether customer_updates['customer_id'] contains duplicates. Return the complete rows for duplicated customer IDs, including all occurrences.

In [15]:
duplicates = customer_updates[
    customer_updates['customer_id'].duplicated(keep=False)
]

display(duplicates)

,customer_id,city,membership
1,102,Mumbai,Silver
2,102,Mumbai,Gold


#### Q2. Understand many-to-many merges

Merge customer_updates with itself on customer_id. Examine customer 102 and explain why the merge creates multiple combinations.

In [22]:
customer_updates

,customer_id,city,membership
0,101,Delhi,Gold
1,102,Mumbai,Silver
2,102,Mumbai,Gold
3,104,Pune,Bronze
4,106,Jaipur,Silver


In [26]:
customer_merge=pd.merge(customer_updates,customer_updates,
                       on='customer_id',
                       suffixes=('_left', '_right'))
customer_merge

,customer_id,city_left,membership_left,city_right,membership_right
0,101,Delhi,Gold,Delhi,Gold
1,102,Mumbai,Silver,Mumbai,Silver
2,102,Mumbai,Silver,Mumbai,Gold
3,102,Mumbai,Gold,Mumbai,Silver
4,102,Mumbai,Gold,Mumbai,Gold
5,104,Pune,Bronze,Pune,Bronze
6,106,Jaipur,Silver,Jaipur,Silver


In [28]:
customer_102 = customer_merge[
    customer_merge['customer_id'] == 102
]

display(customer_102)

,customer_id,city_left,membership_left,city_right,membership_right
1,102,Mumbai,Silver,Mumbai,Silver
2,102,Mumbai,Silver,Mumbai,Gold
3,102,Mumbai,Gold,Mumbai,Silver
4,102,Mumbai,Gold,Mumbai,Gold


#### Q3. Validate a merge

Attempt to merge customers with customer_updates using validate='one_to_one'. Explain the error and identify which dataset violates the expected relationship.

In [35]:
customer_merge=pd.merge(customers,customer_updates,
                       on='customer_id',
                       how='inner',
                       validate='one_to_one')
customer_merge

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge

validate='one_to_one' requires customer_id to be unique in both customers and customer_updates.

#### Q4. Validate a one-to-many relationship

Merge customers with orders using validate='one_to_many'. Confirm whether the relationship is valid and explain what the validation checks.

In [39]:
customer_order=pd.merge(customers,orders,
                       on='customer_id',
                       how='inner',
                       validate='one_to_many')
customer_order

,customer_id,customer_name,city,order_id,sales,status
0,101,Aman,Delhi,1001,2500,Completed
1,101,Aman,Delhi,1003,3200,Completed
2,102,Riya,Mumbai,1002,1800,Cancelled
3,103,Kabir,Delhi,1006,2700,Cancelled
4,104,Neha,Pune,1004,1500,Completed


#### Q5. Handle overlapping column names

Merge product_sales and product_info on product_id, using suffixes ('_sales', '_info'). Inspect the category columns and identify any inconsistencies.

In [44]:
product_merge=pd.merge(product_sales,product_info,
                      on='product_id',
                      how='inner',
                      suffixes=('_sales', '_info'))

product_merge                

,product_id,category_sales,sales,product_name,category_info
0,1,Electronics,2500,Laptop,Electronics
1,1,Electronics,3200,Laptop,Electronics
2,2,Clothing,1800,Shirt,Clothing
3,3,Books,400,Book,Books
4,3,Books,700,Book,Books


#### Q6. Prevent row multiplication

Before merging customer_updates with customers, create a cleaned version of customer_updates containing one row per customer. Keep the latest occurrence based on the existing row order, and reset the index.

In [49]:
cleaned_updates = (
    customer_updates
    .drop_duplicates(subset='customer_id', keep='last')
    .reset_index(drop=True)
)
display(customer_updates)
display(cleaned_updates)

,customer_id,city,membership
0,101,Delhi,Gold
1,102,Mumbai,Silver
2,102,Mumbai,Gold
3,104,Pune,Bronze
4,106,Jaipur,Silver


,customer_id,city,membership
0,101,Delhi,Gold
1,102,Mumbai,Gold
2,104,Pune,Bronze
3,106,Jaipur,Silver


In [51]:
customer_merge = pd.merge(
    customers,
    cleaned_updates,
    on='customer_id',
    how='inner',
    validate='one_to_one'
)

display(customer_merge)

,customer_id,customer_name,city_x,city_y,membership
0,101,Aman,Delhi,Delhi,Gold
1,102,Riya,Mumbai,Mumbai,Gold
2,104,Neha,Pune,Pune,Bronze


#### Q7. Find unmatched customer records

Use a merge indicator to identify customer IDs in customer_updates that do not exist in customers.

In [56]:
customer_merge = pd.merge(
    customers,
    cleaned_updates,
    on='customer_id',
    how='left',
    validate='one_to_one',
    indicator=True
)

only_in_updates=customer_merge[customer_merge['_merge']=='left_only']
only_in_updates

,customer_id,customer_name,city_x,city_y,membership,_merge
2,103,Kabir,Delhi,NaN,NaN,left_only
4,105,Arjun,Mumbai,NaN,NaN,left_only


#### Q8. Find missing product information

Merge product_sales with product_info using a left join. Identify sales records that have no matching product information.

In [61]:
product_merge=pd.merge(product_sales,product_info,
                      on='product_id',
                      how='left',
                       indicator=True
                      )

display(product_merge[product_merge['_merge']=='left_only'])


,product_id,category_x,sales,product_name,category_y,_merge


#### Q9. Compare row counts

Merge product_sales with product_info and compare the number of rows before and after merging. Explain why row counts can increase when the right-hand DataFrame has duplicate keys.

In [76]:
product_merge=pd.merge(product_sales,product_info,
                      on='product_id',
                      how='inner',
                       indicator=True
                      )
product_merge

,product_id,category_x,sales,product_name,category_y,_merge
0,1,Electronics,2500,Laptop,Electronics,both
1,1,Electronics,3200,Laptop,Electronics,both
2,2,Clothing,1800,Shirt,Clothing,both
3,3,Books,400,Book,Books,both
4,3,Books,700,Book,Books,both


In [68]:
print("Rows in product_sales:", len(product_sales))
print("Rows in product_info:", len(product_info))
print("Rows after merging:", len(product_merge))

Rows in product_sales: 5
Rows in product_info: 4
Rows after merging: 5


#### Q10. Build a reliable sales report

Create a category-level sales report by merging product_sales with product_info. The report should contain:
- category
- transaction_count
- total_sales
- average_sales
Ensure that the merge does not unintentionally inflate the sales totals. Explain the validation or cleaning strategy you used.

In [79]:
product_merge=pd.merge(product_sales,product_info,
                      on='product_id',
                      how='inner',
                       indicator=True,
                       validate='many_to_one'
                      )
product_merge

,product_id,category_x,sales,product_name,category_y,_merge
0,1,Electronics,2500,Laptop,Electronics,both
1,1,Electronics,3200,Laptop,Electronics,both
2,2,Clothing,1800,Shirt,Clothing,both
3,3,Books,400,Book,Books,both
4,3,Books,700,Book,Books,both


In [85]:
duplicate_products = product_info[
    product_info['product_id'].duplicated(keep=False)
]

display(duplicate_products)

,product_id,product_name,category
